In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

TIME_OFFSET = 10800 #ADT to UTC is 3hrs

In [2]:
def entropy_feature(x):
    counts = x.value_counts()
    if len(counts) <= 1:
        return 0.0
    return entropy(counts)

def coeff_var(x):
    mean = x.mean()
    if mean == 0:
        return 0.0
    return x.std() / mean

In [3]:
def process_file(k, input_csv, output_csv):
    print(f"Processing {input_csv}...")

    cols = [
        "frame.time_epoch", "frame.len", "ip.src", "ip.dst",
        "tcp.len", "tcp.analysis.retransmission",
        "mbtcp.trans_id", "mbtcp.unit_id", "mbtcp.len",
        "modbus.func_code", "modbus.exception_code", "modbus.reference_num",
        "modbus.regval_uint16", "modbus.byte_cnt", "modbus.word_cnt",
        "modbus.response_time", "modbus.data",
    ]

    dtypes = {
        "frame.len":                   "float32",
        "tcp.len":                     "float32",
        "mbtcp.len":                   "float32",
        "modbus.func_code":            "float32",
        "modbus.byte_cnt":             "float32",
        "modbus.word_cnt":             "float32",
        "modbus.regval_uint16":        "float32",
        "modbus.response_time":        "float32",
        "tcp.analysis.retransmission": "float32",
        "ip.src":                      "category",
        "ip.dst":                      "category",
        "mbtcp.unit_id":               "float32",
        "mbtcp.trans_id":              "float32",
    }

    # ── Pass 1: discover known_ips and observed_windows ────────────────────────
    print("Pass 1: scanning for IPs and time windows...")
    known_ips        = set()
    observed_windows = set()

    for chunk in pd.read_csv(input_csv,
                             usecols=["frame.time_epoch", "ip.src", "ip.dst"],
                             dtype={"ip.src": "category", "ip.dst": "category"},
                             chunksize=500_000):
        chunk = chunk.dropna(subset=["ip.src", "ip.dst"])
        chunk["time_window"] = ((chunk["frame.time_epoch"] + TIME_OFFSET) * k).astype(int)
        known_ips        |= set(chunk["ip.src"].astype(str).unique()) | set(chunk["ip.dst"].astype(str).unique())
        observed_windows |= set(chunk["time_window"].unique())

    known_ips        = sorted(known_ips)
    observed_windows = pd.Index(sorted(observed_windows), name="time_window")
    print(f"IPs found: {known_ips}")
    print(f"Total time windows: {len(observed_windows)}")

    # ── Per-IP accumulators ────────────────────────────────────────────────────
    # Each accumulator is a df indexed by observed_windows, summing across chunks.
    # Means computed by summing are approximate but acceptable for anomaly detection.
    ip_accumulators = {ip: None for ip in known_ips}

    # ── Pass 2: chunked aggregation ────────────────────────────────────────────
    print("Pass 2: aggregating chunks...")
    chunk_idx = 0

    for chunk in pd.read_csv(input_csv, usecols=cols, dtype=dtypes, chunksize=500_000):
        chunk_idx += 1
        print(f"  chunk {chunk_idx}...", end="\r")

        # Type coercion
        chunk["frame.time_epoch"]  = pd.to_numeric(chunk["frame.time_epoch"], errors="coerce")
        chunk["modbus.exception_code"] = pd.to_numeric(
            chunk.get("modbus.exception_code", pd.Series(dtype=float)), errors="coerce"
        )

        # Drop malformed rows
        chunk = chunk.dropna(subset=["ip.src", "ip.dst", "tcp.len"])

        # True zero fills
        chunk["tcp.analysis.retransmission"] = chunk["tcp.analysis.retransmission"].fillna(0)
        chunk["modbus.response_time"]        = chunk["modbus.response_time"].fillna(0)

        # FC-conditional fills
        chunk["modbus.reference_num"]  = chunk["modbus.reference_num"].fillna(0)
        chunk["modbus.regval_uint16"]  = chunk["modbus.regval_uint16"].fillna(0)
        chunk["modbus.word_cnt"]       = chunk["modbus.word_cnt"].fillna(0)
        chunk["modbus.byte_cnt"]       = chunk["modbus.byte_cnt"].fillna(0)
        chunk["modbus.data"]           = chunk["modbus.data"].fillna("")

        # Derived columns — Modbus rows only
        modbus_mask = chunk["modbus.func_code"].notna()
        chunk["is_stacked"]   = 0.0
        chunk["stack_excess"] = 0.0
        chunk["len_mismatch"] = 0.0
        chunk["is_mismatch"]  = 0.0

        chunk.loc[modbus_mask, "is_stacked"]   = (
            chunk.loc[modbus_mask, "tcp.len"] > (chunk.loc[modbus_mask, "mbtcp.len"] + 6)
        ).astype(float)
        chunk.loc[modbus_mask, "stack_excess"] = (
            chunk.loc[modbus_mask, "tcp.len"] - (chunk.loc[modbus_mask, "mbtcp.len"] + 6)
        ).clip(lower=0)
        chunk.loc[modbus_mask, "len_mismatch"] = (
            chunk.loc[modbus_mask, "mbtcp.len"] - chunk.loc[modbus_mask, "modbus.byte_cnt"]
        )
        chunk.loc[modbus_mask, "is_mismatch"]  = (
            chunk.loc[modbus_mask, "len_mismatch"].abs() > 2
        ).astype(float)

        chunk["is_duplicate"] = chunk.duplicated(
            subset=["ip.src", "ip.dst", "mbtcp.trans_id", "modbus.data"], keep=False
        ).astype(float)

        # Timestamp & windowing
        chunk["timestamp"]   = chunk["frame.time_epoch"] + TIME_OFFSET
        chunk["time_window"] = (chunk["timestamp"] * k).astype(int)

        # IAT within (time_window, src, dst)
        chunk = chunk.sort_values("timestamp")
        chunk["iat"] = (
            chunk.groupby(["time_window", "ip.src", "ip.dst"])["timestamp"]
            .diff()
            .fillna(0)
        )

        # Per-IP aggregation
        for ip in known_ips:
            tx = chunk[chunk["ip.src"].astype(str) == ip]
            rx = chunk[chunk["ip.dst"].astype(str) == ip]

            if len(tx) == 0 and len(rx) == 0:
                continue

            tx_agg = tx.groupby("time_window").agg(
                # Volume
                tx_packet_count       = ("frame.len",                   "count"),
                tx_total_bytes        = ("frame.len",                   "sum"),
                tx_mean_pkt_size      = ("frame.len",                   "mean"),
                tx_std_pkt_size       = ("frame.len",                   "std"),
                # IAT
                tx_iat_mean           = ("iat",                         "mean"),
                tx_iat_std            = ("iat",                         "std"),
                tx_iat_cv             = ("iat",                         coeff_var),
                # Destinations & slaves
                tx_unique_dst         = ("ip.dst",                      "nunique"),
                tx_unique_unit_ids    = ("mbtcp.unit_id",               "nunique"),
                # Function codes
                tx_unique_fc          = ("modbus.func_code",            "nunique"),
                tx_func_entropy       = ("modbus.func_code",            entropy_feature),
                tx_read_count         = ("modbus.func_code",            lambda x: x.isin([3, 4]).sum()),
                tx_write_count        = ("modbus.func_code",            lambda x: x.isin([5, 6, 15, 16]).sum()),
                tx_fc3                = ("modbus.func_code",            lambda x: (x == 3).sum()),
                tx_fc4                = ("modbus.func_code",            lambda x: (x == 4).sum()),
                tx_fc5                = ("modbus.func_code",            lambda x: (x == 5).sum()),
                tx_fc6                = ("modbus.func_code",            lambda x: (x == 6).sum()),
                tx_fc15               = ("modbus.func_code",            lambda x: (x == 15).sum()),
                tx_fc16               = ("modbus.func_code",            lambda x: (x == 16).sum()),
                # Exceptions
                tx_exception_count    = ("modbus.exception_code",       lambda x: x.notna().sum()),
                # Register addressing
                tx_unique_regs        = ("modbus.reference_num",        "nunique"),
                tx_register_std       = ("modbus.reference_num",        "std"),
                tx_register_entropy   = ("modbus.reference_num",        entropy_feature),
                # Register values
                tx_regval_mean        = ("modbus.regval_uint16",        "mean"),
                tx_regval_std         = ("modbus.regval_uint16",        "std"),
                # Payload sizes
                tx_mean_mbap_len      = ("mbtcp.len",                   "mean"),
                tx_byte_cnt_mean      = ("modbus.byte_cnt",             "mean"),
                tx_word_cnt_mean      = ("modbus.word_cnt",             "mean"),
                # Response timing
                tx_resp_time_mean     = ("modbus.response_time",        "mean"),
                tx_resp_time_std      = ("modbus.response_time",        "std"),
                # Retransmissions
                tx_retransmission_sum = ("tcp.analysis.retransmission", "sum"),
                # Stacking
                tx_stacked_count      = ("is_stacked",                  "sum"),
                tx_mean_stack_excess  = ("stack_excess",                "mean"),
                # Length mismatch
                tx_len_mismatch_mean  = ("len_mismatch",                "mean"),
                tx_len_mismatch_count = ("is_mismatch",                 "sum"),
                # Duplicate / replay
                tx_duplicate_count    = ("is_duplicate",                "sum"),
                # Transaction IDs
                tx_unique_tids        = ("mbtcp.trans_id",              "nunique"),
            )

            rx_agg = rx.groupby("time_window").agg(
                rx_packet_count       = ("frame.len",                   "count"),
                rx_total_bytes        = ("frame.len",                   "sum"),
                rx_unique_src         = ("ip.src",                      "nunique"),
                rx_unique_unit_ids    = ("mbtcp.unit_id",               "nunique"),
                rx_retransmission_sum = ("tcp.analysis.retransmission", "sum"),
                rx_resp_time_mean     = ("modbus.response_time",        "mean"),
            )

            ip_agg = tx_agg.join(rx_agg, how="outer")
            ip_agg = ip_agg.reindex(observed_windows, fill_value=0).fillna(0)

            if ip_accumulators[ip] is None:
                ip_accumulators[ip] = ip_agg
            else:
                ip_accumulators[ip] = ip_accumulators[ip].add(ip_agg, fill_value=0)

    # ── Finalize: derived ratios and join all IPs ──────────────────────────────
    print("\nFinalizing...")
    result = pd.DataFrame(index=observed_windows)

    for ip in known_ips:
        ip_df = ip_accumulators[ip]
        if ip_df is None:
            continue

        pkt = ip_df["tx_packet_count"].to_numpy(dtype=float)

        def safe_div(num_col):
            num = ip_df[num_col].to_numpy(dtype=float)
            return np.divide(num, pkt, out=np.zeros_like(pkt), where=pkt > 0)

        ip_df["tx_write_ratio"]     = safe_div("tx_write_count")
        ip_df["tx_read_ratio"]      = safe_div("tx_read_count")
        ip_df["tx_exception_ratio"] = safe_div("tx_exception_count")
        ip_df["tx_dup_tids"]        = ip_df["tx_packet_count"] - ip_df["tx_unique_tids"]
        ip_df["tx_tid_ratio"]       = safe_div("tx_unique_tids")
        ip_df["tx_regval_delta"]    = ip_df["tx_regval_mean"].diff().fillna(0)

        safe_ip = ip.replace(".", "_")
        ip_df   = ip_df.add_prefix(f"{safe_ip}__")
        result  = result.join(ip_df, how="left")

    result = result.fillna(0).reset_index()
    result.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}")

In [ ]:
process_file(1, "../train/benign_nw_analysis.csv", "../train/benign_flows_kit.csv")

Processing ../train/benign_nw_analysis.csv...
Pass 1: scanning for IPs and time windows...


KeyboardInterrupt: 

In [ ]:
process_file(1, "../train/cscada_attack_ssw_kit.csv", "../train/cscada_flows_kit.csv")

Processing ../train/cscada_attack_ssw_analysis.csv...


/tmp/ipykernel_670463/4246770210.py:7: DtypeWarning: Columns (0: mbtcp.trans_id, 1: mbtcp.unit_id, 2: mbtcp.len, 3: mbtcp.prot_id, 4: modbus.func_code, 5: modbus.reference_num, 6: modbus.regnum16, 7: modbus.bitnum, 8: modbus.word_cnt, 9: modbus.bit_cnt, 10: modbus.regval_uint16, 11: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8']


/tmp/ipykernel_670463/4246770210.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    14960.000000
mean        77.215775
std        144.355317
min          1.000000
25%         45.000000
50%         50.000000
75%         90.000000
max       8241.000000
dtype: float64
Saved → ../train/1s_cscada_flows.csv



In [ ]:
process_file(1, "../train/ext_attack_nw_kit.csv", "../train/external_flows_kit.csv")

Processing ../train/ext_attack_nw_analysis.csv...


/tmp/ipykernel_670463/4246770210.py:7: DtypeWarning: Columns (0: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.7', '185.175.0.8']


/tmp/ipykernel_670463/4246770210.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count     1877.000000
mean       291.241343
std       1253.904741
min          1.000000
25%         45.000000
50%         45.000000
75%         51.000000
max      20281.000000
dtype: float64
Saved → ../train/1s_external_flows.csv



In [ ]:
process_file(1, "../train/benign_nw_kit.csv", "../train/benign_flows_kit.csv")

Index(['time_window', '185_175_0_3__tx_packet_count',
       '185_175_0_3__tx_total_bytes', '185_175_0_3__tx_mean_pkt_size',
       '185_175_0_3__tx_std_pkt_size', '185_175_0_3__tx_iat_mean',
       '185_175_0_3__tx_iat_std', '185_175_0_3__tx_iat_cv',
       '185_175_0_3__tx_unique_dst', '185_175_0_3__tx_unique_unit_ids',
       ...
       '185_175_0_8__rx_unique_src', '185_175_0_8__rx_unique_unit_ids',
       '185_175_0_8__rx_retransmission_sum', '185_175_0_8__rx_resp_time_mean',
       '185_175_0_8__tx_write_ratio', '185_175_0_8__tx_read_ratio',
       '185_175_0_8__tx_exception_ratio', '185_175_0_8__tx_dup_tids',
       '185_175_0_8__tx_tid_ratio', '185_175_0_8__tx_regval_delta'],
      dtype='str', length=246)


In [ ]:
# process_file(10, "../train/benign_nw.csv", "../train/100ms_benign_flows.csv") #this is for one second windows

In [ ]:
# process_file(10, "../train/cscada_attack_ssw.csv", "../train/100ms_cscada_flows.csv")

In [ ]:
# process_file(10, "../train/ext_attack_nw.csv", "../train/100ms_external_flows.csv")

In [ ]:
import os
os.system('notify-send "Python Script" "Execution complete!"')

0